# Day 7 — Advanced Training Techniques

**Agri-AI Bootcamp** · Phase 2 of 5: *Deep Learning & PyTorch* · Days 5–8

> Add nn.BatchNorm1d and nn.Dropout(0.3) to the Day 6 model; train 50 epochs and compare curves.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

Adam gives each weight its own adaptive step size and is a strong default; SGD with momentum can generalise well but needs tuning. Dropout switches off random neurons during training so the network can’t over-rely on one. BatchNorm normalises layer inputs for faster, steadier training.

### 🌾 Agriculture example

On a small soil dataset the plain MLP’s validation accuracy zig-zags; with BatchNorm1d and Dropout(0.3) the curve smooths out sooner.

> **Watch out:** Use model.train() and model.eval(); dropout and BatchNorm behave differently in each mode.

## 2. Setup

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
dev = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0)

## 3. Data

In [ ]:
# ---- Small, noisy soil dataset (small on purpose: regularisation shows up) ---
rng = np.random.default_rng(1)
CLASSES = ["Clay", "Sand", "Loam", "Silt"]
centres = {"Clay": [25,25,50,6.2,2.8], "Sand": [80,12,8,6.8,0.9],
           "Loam": [42,38,20,6.5,3.4], "Silt": [15,70,15,6.9,2.1]}
rows, labels = [], []
for k, c in enumerate(CLASSES):
    rows.append(rng.normal(centres[c], [11,11,10,.6,1.1], (60, 5)))   # only 60/class
    labels += [k]*60
Xs = np.vstack(rows).astype(np.float32); ys = np.array(labels)
p = rng.permutation(len(ys)); Xs, ys = Xs[p], ys[p]
cut = int(.7*len(ys)); mu, sd = Xs[:cut].mean(0), Xs[:cut].std(0)
Xtr = torch.tensor((Xs[:cut]-mu)/sd, device=dev); ytr = torch.tensor(ys[:cut], device=dev)
Xte = torch.tensor((Xs[cut:]-mu)/sd, device=dev); yte = torch.tensor(ys[cut:], device=dev)
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape))

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
net=nn.Sequential(nn.Linear(5,32),nn.BatchNorm1d(32),nn.ReLU(),
                  nn.Dropout(0.3),nn.Linear(32,4))
opt=torch.optim.Adam(net.parameters(),1e-3)
net.train()   # net.eval() when validating
```

## 5. Worked solution

In [ ]:
# ---- Three models: plain / +BatchNorm+Dropout / +both with SGD-momentum -----
def plain():
    return nn.Sequential(nn.Linear(5,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,4))
def regd():
    return nn.Sequential(nn.Linear(5,64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
                         nn.Linear(64,32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.3),
                         nn.Linear(32,4))

def run(make, opt_name="adam", epochs=300, bs=32):
    torch.manual_seed(0)
    net = make().to(dev); lossf = nn.CrossEntropyLoss()
    opt = (torch.optim.Adam(net.parameters(), 1e-3) if opt_name == "adam"
           else torch.optim.SGD(net.parameters(), 1e-2, momentum=0.9))
    tr_acc, te_acc = [], []
    for ep in range(epochs):
        net.train()                                   # dropout ON, BN uses batch stats
        perm = torch.randperm(len(Xtr), device=dev)
        for i in range(0, len(perm), bs):
            idx = perm[i:i+bs]
            if len(idx) < 2: continue                 # BatchNorm needs >1 sample
            opt.zero_grad(); lossf(net(Xtr[idx]), ytr[idx]).backward(); opt.step()
        net.eval()                                    # dropout OFF, BN uses running stats
        with torch.no_grad():
            tr_acc.append((net(Xtr).argmax(1) == ytr).float().mean().item())
            te_acc.append((net(Xte).argmax(1) == yte).float().mean().item())
    return net, np.array(tr_acc), np.array(te_acc)

runs = {"plain + Adam":      run(plain, "adam"),
        "BN+Dropout + Adam": run(regd,  "adam"),
        "BN+Dropout + SGD-m":run(regd,  "sgd")}

plt.figure(figsize=(11, 3.8))
for i, (name, (net, tr, te)) in enumerate(runs.items()):
    gap = tr[-20:].mean() - te[-20:].mean()
    print(f"{name:<20} train {tr[-20:].mean():.3f} | test {te[-20:].mean():.3f} | gap {gap:+.3f}"
          f" | test wobble (std of last 50) {te[-50:].std():.3f}")
    plt.subplot(1, 3, i+1); plt.plot(tr, label="train"); plt.plot(te, label="test")
    plt.title(name, fontsize=9); plt.ylim(0, 1.02); plt.xlabel("epoch")
    if i == 0: plt.legend()
plt.tight_layout(); plt.show()

# ---- train() vs eval() is not cosmetic --------------------------------------
net = runs["BN+Dropout + Adam"][0]
net.train()
with torch.no_grad(): a = (net(Xte).argmax(1) == yte).float().mean().item()
net.eval()
with torch.no_grad(): b = (net(Xte).argmax(1) == yte).float().mean().item()
print(f"\nsame weights, same data: train-mode acc {a:.3f} vs eval-mode acc {b:.3f}")
print("Forgetting net.eval() is one of the most common silent bugs in PyTorch.")

## 6. Your turn

1. Sweep dropout p over 0.0, 0.1, 0.3, 0.5 and plot the final test accuracy.
2. Remove BatchNorm but keep Dropout — which of the two smooths the curve more here?
3. Add a learning-rate scheduler (`torch.optim.lr_scheduler.StepLR`) and watch the late epochs.

In [ ]:
# TODO 1: dropout sweep
def regd_p(p):
    return lambda: nn.Sequential(nn.Linear(5,64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p),
                                 nn.Linear(64,32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(p),
                                 nn.Linear(32,4))
for p in [0.0, 0.1, 0.3, 0.5]:
    _, tr, te = run(regd_p(p), "adam", epochs=200)
    print(f"dropout {p}: train {tr[-20:].mean():.3f} test {te[-20:].mean():.3f}")

# TODO 2: Dropout only (no BatchNorm)
# TODO 3: StepLR scheduler

## 7. Dataset and references

**Dataset:** [Simulate soil data with NumPy (sand, silt, clay %, pH, organic matter)](https://numpy.org/doc/stable/reference/random/generator.html)

**References**

- [CS231n Neural Networks Part 2](https://cs231n.github.io/neural-networks-2/)

**Course material:** Stanford CS231n: Neural Networks Part 2

---
**Next:** Day 8 — PyTorch Data Pipelines